In [1]:
!pip install -U openai pandas numpy tqdm python-dotenv

     |████████████████████████████████| 1.3 MB 5.3 MB/s            
  Attempting uninstall: openai
    Found existing installation: openai 2.37.0
    Uninstalling openai-2.37.0:
      Successfully uninstalled openai-2.37.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openai 0.3.38 requires openai<2.0.0,>=1.66.3, but you have openai 2.38.0 which is incompatible.


In [3]:
from __future__ import annotations

import os
import re
import gc
import json
import time
import uuid
import pickle
import hashlib
import datetime as dt
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Use environment variable / .env. Do not paste API keys into the notebook.
client = OpenAI(api_key="")

PROVIDER = "openai"
MODEL_NAME = "gpt-5.4"
TEMPERATURE = 1.0

# Planning is not the actual generation manipulation, so keep it low-noise.
PLANNING_TEMPERATURE = 0.0

REASONING_EFFORT = "none"
TEXT_VERBOSITY = "medium"
PROMPT_CACHE_RETENTION = "24h"
PROMPT_CACHE_KEY = None

N_FINAL_PER_TASK_METHOD_STRATEGY = 150
N_STRATA = 5
N_PER_STRATUM = N_FINAL_PER_TASK_METHOD_STRATEGY // N_STRATA
assert N_STRATA * N_PER_STRATUM == N_FINAL_PER_TASK_METHOD_STRATEGY

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

# Existing analysis object from the current final analysis notebook.
EXPERIMENT_DATA_PATH = Path("final_analysis/input_data/experiment_data.pkl")

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]
EXPERIMENT_ID = "openai_followup_simplestrat5_g2css3"

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / EXPERIMENT_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "baseline": DATA_ROOT / "01_loaded_baseline",
    "planning_plans": DATA_ROOT / "02_simplestrat_planning" / "plans",
    "planning_batch_inputs": DATA_ROOT / "02_simplestrat_planning" / "batch_inputs",
    "planning_manifests": DATA_ROOT / "02_simplestrat_planning" / "manifests",
    "planning_raw_outputs": DATA_ROOT / "02_simplestrat_planning" / "raw_outputs",
    "planning_raw_errors": DATA_ROOT / "02_simplestrat_planning" / "raw_errors",
    "planning_parsed": DATA_ROOT / "02_simplestrat_planning" / "parsed",
    "g2_processing": DATA_ROOT / "03_g2_css_processing",
    "round2_plans": DATA_ROOT / "04_round2_new_baselines" / "plans",
    "round2_batch_inputs": DATA_ROOT / "04_round2_new_baselines" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "04_round2_new_baselines" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "04_round2_new_baselines" / "raw_outputs",
    "round2_raw_errors": DATA_ROOT / "04_round2_new_baselines" / "raw_errors",
    "round2_parsed": DATA_ROOT / "04_round2_new_baselines" / "parsed",
    "compiled": DATA_ROOT / "05_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636


In [4]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 24) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def normalize_embeddings(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(f"None of these candidate columns found: {candidates}\nAvailable columns:\n{df.columns.tolist()}")

In [5]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_smartphone",
        "task_family": "slogan",
        "task_label": "Smartphone slogan",
        "task_prompt_key": "smartphone",
    },
    {
        "task_id": "slogan_soda",
        "task_family": "slogan",
        "task_label": "Soda slogan",
        "task_prompt_key": "soda",
    },
    {
        "task_id": "slogan_blood_donation",
        "task_family": "slogan",
        "task_label": "Blood donation slogan",
        "task_prompt_key": "blood_donation",
    },
    {
        "task_id": "aut_shoe",
        "task_family": "aut",
        "task_label": "AUT: shoe",
        "task_prompt_key": "shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "task_id": "aut_button",
        "task_family": "aut",
        "task_label": "AUT: button",
        "task_prompt_key": "button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "task_id": "aut_key",
        "task_family": "aut",
        "task_label": "AUT: key",
        "task_prompt_key": "key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "task_id": "aut_wooden_pencil",
        "task_family": "aut",
        "task_label": "AUT: wooden pencil",
        "task_prompt_key": "wooden_pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "task_id": "aut_automobile_tire",
        "task_family": "aut",
        "task_label": "AUT: automobile tire",
        "task_prompt_key": "automobile_tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
    {
        "task_id": "story_jungle",
        "task_family": "story",
        "task_label": "Story: jungle",
        "task_prompt_key": "jungle",
    },
    {
        "task_id": "story_parachute",
        "task_family": "story",
        "task_label": "Story: parachute",
        "task_prompt_key": "parachute",
    },
    {
        "task_id": "story_horror",
        "task_family": "story",
        "task_label": "Story: horror",
        "task_prompt_key": "horror",
    },
    {
        "task_id": "story_life_last_seconds",
        "task_family": "story",
        "task_label": "Story: life / last seconds",
        "task_prompt_key": "life_last_seconds",
    },
]

TASK_BY_ID = {t["task_id"]: t for t in TASK_SETTINGS}
TASK_ORDER = [t["task_id"] for t in TASK_SETTINGS]
STRATEGIES = ["vanilla", "diverge"]

SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_shoe", "aut_button", "aut_key", "aut_wooden_pencil", "aut_automobile_tire"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def round2_final_line_for_context(strategy: str, context_type: str) -> str:
    """
    Keep R2 style as close as possible to the original.

    For contexts with prior responses, use the original previous-response wording.
    For SimpleStrat, there are no prior responses, so the diverge line mirrors the original
    "stand out" intent without falsely saying previous responses were shown.
    """
    if strategy == "vanilla":
        return "Now generate one new response for the same task."

    if strategy == "diverge" and context_type == "prior_responses":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    if strategy == "diverge" and context_type == "stratum":
        return (
            "Now generate one new response for the same task. "
            "It should stand out from other responses that might be generated for this same task while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy/context_type: {strategy}, {context_type}")

In [6]:
run_config = {
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "temperature": TEMPERATURE,
    "planning_temperature": PLANNING_TEMPERATURE,
    "reasoning_effort": REASONING_EFFORT,
    "text_verbosity": TEXT_VERBOSITY,
    "prompt_cache_retention": PROMPT_CACHE_RETENTION,
    "prompt_cache_key": PROMPT_CACHE_KEY,
    "n_final_per_task_method_strategy": N_FINAL_PER_TASK_METHOD_STRATEGY,
    "n_strata": N_STRATA,
    "n_per_stratum": N_PER_STRATUM,
    "task_order": TASK_ORDER,
    "strategies": STRATEGIES,
    "data_root": str(DATA_ROOT),
    "experiment_data_path": str(EXPERIMENT_DATA_PATH),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/00_metadata/experiment_config__20260523_124432__a2167636.json')

In [7]:
@dataclass
class ExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None


@dataclass
class ProviderExperimentData:
    long_df: Optional[pd.DataFrame] = None
    embeddings: Optional[np.ndarray] = None
    provider: Optional[str] = None
    provider_label: Optional[str] = None
    model: Optional[str] = None


class GenericPicklePlaceholder:
    def __init__(self, *args, **kwargs):
        self.__dict__.update(kwargs)

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["state"] = state


class CompatibleUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "__main__":
            if name == "ExperimentData":
                return ExperimentData
            if name == "ProviderExperimentData":
                return ProviderExperimentData
            return GenericPicklePlaceholder
        return super().find_class(module, name)


def load_experiment_data(path: Path) -> ExperimentData:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run the final analysis setup first, or update EXPERIMENT_DATA_PATH."
        )

    with open(path, "rb") as f:
        obj = CompatibleUnpickler(f).load()

    if hasattr(obj, "long_df") and hasattr(obj, "embeddings") and obj.long_df is not None and obj.embeddings is not None:
        return obj

    candidate_attrs = getattr(obj, "__dict__", {})
    provider_objects = []

    for attr_value in candidate_attrs.values():
        if isinstance(attr_value, dict):
            provider_objects.extend(
                [
                    v for v in attr_value.values()
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )
        elif isinstance(attr_value, (list, tuple)):
            provider_objects.extend(
                [
                    v for v in attr_value
                    if hasattr(v, "long_df") and hasattr(v, "embeddings")
                    and v.long_df is not None and v.embeddings is not None
                ]
            )

    if provider_objects:
        long_parts = [p.long_df for p in provider_objects]
        emb_parts = [np.asarray(p.embeddings) for p in provider_objects]
        combined = ExperimentData(
            long_df=pd.concat(long_parts, ignore_index=True, sort=False),
            embeddings=np.vstack(emb_parts),
        )
        if len(combined.long_df) != combined.embeddings.shape[0]:
            raise RuntimeError("Concatenated long_df and embeddings row counts do not match.")
        return combined

    raise RuntimeError("Loaded pickle object does not contain usable long_df and embeddings.")


experiment_data = load_experiment_data(EXPERIMENT_DATA_PATH)

long_df = experiment_data.long_df.copy()
long_df["_row_pos"] = np.arange(len(long_df))

if len(long_df) != experiment_data.embeddings.shape[0]:
    raise RuntimeError(
        f"long_df rows and embeddings rows do not match: {len(long_df)} vs {experiment_data.embeddings.shape[0]}"
    )

TEXT_COL = first_existing_col(
    long_df,
    ["text", "response_text", "output_text", "model_output", "clean_text", "idea", "response"],
)

print("Loaded experiment data:")
print("long_df:", long_df.shape)
print("embeddings:", experiment_data.embeddings.shape)
print("Using text column:", TEXT_COL)
print("Providers:", sorted(long_df["provider"].astype(str).unique()))

Loaded experiment data:
long_df: (64800, 53)
embeddings: (64800, 768)
Using text column: text
Providers: ['anthropic', 'gemini', 'openai']


In [8]:
baseline_mask = (
    long_df["provider"].astype(str).eq(PROVIDER)
    & long_df["round"].astype(int).eq(1)
    & long_df["strategy"].astype(str).eq("vanilla")
    & long_df["condition"].astype(str).eq("base")
    & long_df["task_id"].astype(str).isin(TASK_ORDER)
)

baseline_df = long_df.loc[baseline_mask].copy()

baseline_df["baseline_text"] = baseline_df[TEXT_COL].map(clean_model_text)
baseline_df = baseline_df.sort_values(["task_id", "group_id", "agent_id"]).reset_index(drop=True)

counts = (
    baseline_df
    .groupby("task_id", observed=True)
    .agg(
        n=("baseline_text", "size"),
        n_nonempty=("baseline_text", lambda x: x.notna().sum()),
        task_family=("task_family", "first"),
    )
    .reset_index()
    .sort_values("task_id")
)

display(counts)

missing_tasks = sorted(set(TASK_ORDER) - set(counts["task_id"]))
bad_counts = counts[counts["n"] != N_FINAL_PER_TASK_METHOD_STRATEGY]

if missing_tasks:
    raise RuntimeError(f"Missing tasks in baseline data: {missing_tasks}")

if not bad_counts.empty:
    raise RuntimeError(f"Expected 150 baseline rows per task. Bad counts:\n{bad_counts}")

if baseline_df["baseline_text"].isna().any() or baseline_df["baseline_text"].eq("").any():
    bad = baseline_df[baseline_df["baseline_text"].isna() | baseline_df["baseline_text"].eq("")]
    raise RuntimeError(f"Empty baseline text rows found:\n{bad.head()}")

baseline_light_cols = [
    "_row_pos", "provider", "provider_label", "model", "round", "task_id", "task_label",
    "task_family", "task_family_label", "strategy", "condition", "group_id", "agent_id",
    "agent_index", "baseline_text"
]
baseline_light_cols = [c for c in baseline_light_cols if c in baseline_df.columns]

baseline_light = baseline_df[baseline_light_cols].copy()

baseline_pkl_path = DIRS["baseline"] / "openai_neutral_r1_base_12tasks_150each.pkl"
baseline_csv_path = DIRS["baseline"] / "openai_neutral_r1_base_12tasks_150each.csv"

baseline_light.to_pickle(baseline_pkl_path)
baseline_light.to_csv(baseline_csv_path, index=False)

print("Saved:")
print(baseline_pkl_path)
print(baseline_csv_path)

,task_id,n,n_nonempty,task_family
0,slogan_smartphone,150,150,slogan
1,slogan_soda,150,150,slogan
2,slogan_blood_donation,150,150,slogan
3,aut_shoe,150,150,aut
4,aut_button,150,150,aut
5,aut_key,150,150,aut
6,aut_wooden_pencil,150,150,aut
7,aut_automobile_tire,150,150,aut
8,story_jungle,150,150,story
9,story_parachute,150,150,story


Saved:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/01_loaded_baseline/openai_neutral_r1_base_12tasks_150each.pkl
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/01_loaded_baseline/openai_neutral_r1_base_12tasks_150each.csv


In [13]:
PLANNING_SYSTEM_INSTRUCTIONS = (
    "You identify semantic diversity strata for controlled text-generation experiments. "
    "Return valid JSON only. Do not include markdown fences or commentary."
)


def build_simplestrat_planning_prompt(task: dict) -> str:
    return f"""
We will later generate 150 independent responses to the following task.

Task prompt:
{base_task_prompt(task)}

Identify exactly {N_STRATA} mutually distinct semantic strata for valid responses to this task.

Use the following procedure internally before choosing the final strata:
1. Consider questions that would separate the space of possible valid responses into broad, meaningfully different groups.
2. Prefer distinctions that would split the possible valid responses into reasonably balanced groups, rather than isolating rare edge cases.
3. Convert the best distinctions into categorical conceptual directions for generation.
4. Exclude distinctions based only on superficial wording, tone, length, punctuation, formatting, or synonyms.
5. Exclude strata that name a specific candidate answer, force a specific phrase, or make the original task harder to satisfy.

The final strata must satisfy all of these requirements:
- Each stratum must be a semantic/content direction, not a superficial style change.
- Each stratum must be broad enough to support many different valid responses.
- The strata must be mutually distinct enough that responses generated under different strata are likely to differ conceptually.
- The strata must collectively cover a wide range of plausible valid responses to the task.
- The generation_instruction must be concise and usable as an added constraint in a later generation prompt.

Return JSON only with this exact structure:
{{
  "task_id": "...",
  "strata": [
    {{
      "stratum_id": 1,
      "name": "short name",
      "description": "one sentence describing the semantic direction",
      "generation_instruction": "a concise instruction that can be appended to the generation prompt",
      "why_broad": "one short sentence explaining why this stratum can support many valid responses",
      "why_distinct": "one short sentence explaining how this stratum differs from the other strata"
    }}
  ]
}}

The JSON must contain exactly {N_STRATA} strata with stratum_id values 1 through {N_STRATA}.
Do not include any text outside the JSON.
""".strip()


def build_planning_plan() -> pd.DataFrame:
    rows = []
    for task in TASK_SETTINGS:
        request_basis = {
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "experiment_id": EXPERIMENT_ID,
            "stage": "simplestrat_planning",
            "task_id": task["task_id"],
            "task_family": task["task_family"],
            "task_label": task["task_label"],
            "n_strata_requested": N_STRATA,
        }
        request_key = "simplestrat_plan__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

        rows.append({
            **request_basis,
            "request_key": request_key,
            "system_instructions": PLANNING_SYSTEM_INSTRUCTIONS,
            "user_prompt": build_simplestrat_planning_prompt(task),
            "temperature": PLANNING_TEMPERATURE,
            "max_output_tokens": 900,
            "created_at_utc": now_iso(),
        })

    out = pd.DataFrame(rows)
    if out["request_key"].duplicated().any():
        raise RuntimeError("Duplicate planning request_key detected.")
    return out


planning_plan_df = build_planning_plan()

planning_plan_path = DIRS["planning_plans"] / f"simplestrat5_planning_plan__{RUN_ID}.csv"
planning_plan_df.to_csv(planning_plan_path, index=False)

print("Planning requests:", len(planning_plan_df))
display(planning_plan_df[["task_id", "task_family", "request_key"]])
print("Saved:", planning_plan_path)

Planning requests: 12


,task_id,task_family,request_key
0,slogan_smartphone,slogan,simplestrat_plan__7a8a94a4f8512b0cbbb9d148
1,slogan_soda,slogan,simplestrat_plan__d2c6b389f131972a7f2491d3
2,slogan_blood_donation,slogan,simplestrat_plan__b68721debb31fa88502fe69b
3,aut_shoe,aut,simplestrat_plan__45568a6c977cac1900cc18b4
4,aut_button,aut,simplestrat_plan__09b5df154613e79056390f1a
5,aut_key,aut,simplestrat_plan__071480d781689369e96a0b63
6,aut_wooden_pencil,aut,simplestrat_plan__d45045d22e6efd6da228409c
7,aut_automobile_tire,aut,simplestrat_plan__9945be9066e3d34d5c5af9ea
8,story_jungle,story,simplestrat_plan__e4086def005b12e546a5735f
9,story_parachute,story,simplestrat_plan__1548d8879bcdad6832712743


Saved: ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/plans/simplestrat5_planning_plan__20260523_124432__a2167636.csv


In [14]:
def json_safe(obj):
    """Convert numpy/pandas objects into JSON-safe Python objects."""
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)):
            return None
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if (np.isnan(val) or np.isinf(val)) else val
    if isinstance(obj, (np.ndarray,)):
        return [json_safe(x) for x in obj.tolist()]
    if pd.isna(obj) if not isinstance(obj, (list, tuple, dict)) else False:
        return None
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    return str(obj)


def make_openai_responses_batch_jsonl(
    plan_df: pd.DataFrame,
    batch_jsonl_path: Path,
) -> Path:
    if batch_jsonl_path.exists():
        raise FileExistsError(f"Refusing to overwrite: {batch_jsonl_path}")

    batch_jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    with open(batch_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": MODEL_NAME,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            if REASONING_EFFORT is not None:
                body["reasoning"] = {"effort": REASONING_EFFORT}

            if TEXT_VERBOSITY is not None:
                body["text"] = {"verbosity": TEXT_VERBOSITY}

            if PROMPT_CACHE_RETENTION is not None:
                body["prompt_cache_retention"] = PROMPT_CACHE_RETENTION

            if PROMPT_CACHE_KEY is not None:
                body["prompt_cache_key"] = PROMPT_CACHE_KEY

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }

            f.write(json.dumps(json_safe(request), ensure_ascii=False) + "\n")

    print(f"Wrote batch input: {batch_jsonl_path}")
    print(f"Requests: {len(plan_df):,}")
    return batch_jsonl_path


def submit_openai_batch(
    batch_jsonl_path: Path,
    stage_name: str,
    plan_path: Path,
    manifest_dir: Path,
) -> dict:
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "deflect_creativity",
            "experiment_id": EXPERIMENT_ID,
            "stage": stage_name,
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "local_input_file": str(batch_jsonl_path),
            "local_plan_file": str(plan_path),
        },
    )

    batch_info = {
        "run_id": RUN_ID,
        "experiment_id": EXPERIMENT_ID,
        "stage": stage_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "status_at_submission": batch.status,
        "submitted_at_utc": now_iso(),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = manifest_dir / f"{stage_name}__batch_manifest__{batch.id}.json"
    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite manifest: {manifest_path}")

    write_json(manifest_path, json_safe(batch_info))
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted batch:")
    print(json.dumps(json_safe(batch_info), indent=2))
    return batch_info


def check_openai_batch(batch_id: str) -> dict:
    batch = client.batches.retrieve(batch_id)

    info = {
        "batch_id": batch.id,
        "status": batch.status,
        "request_counts": None,
        "output_file_id": batch.output_file_id,
        "error_file_id": batch.error_file_id,
        "created_at": batch.created_at,
        "in_progress_at": getattr(batch, "in_progress_at", None),
        "finalizing_at": getattr(batch, "finalizing_at", None),
        "completed_at": getattr(batch, "completed_at", None),
        "failed_at": getattr(batch, "failed_at", None),
        "expired_at": getattr(batch, "expired_at", None),
        "cancelled_at": getattr(batch, "cancelled_at", None),
    }

    if batch.request_counts:
        info["request_counts"] = {
            "total": batch.request_counts.total,
            "completed": batch.request_counts.completed,
            "failed": batch.request_counts.failed,
        }

    usage = getattr(batch, "usage", None)
    if usage:
        try:
            info["usage"] = usage.model_dump()
        except Exception:
            info["usage"] = str(usage)

    print(json.dumps(json_safe(info), indent=2))
    return info


def download_openai_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    raw_error_dir: Path,
    stage_name: str,
) -> tuple[Optional[Path], Optional[Path]]:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        raise RuntimeError(f"Batch is not completed yet. Current status: {batch.status}")

    output_path = raw_output_dir / f"{stage_name}__{batch_id}__output.jsonl"
    error_path = raw_error_dir / f"{stage_name}__{batch_id}__errors.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite output file: {output_path}")

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")
    else:
        output_path = None
        print("No output file.")

    if batch.error_file_id:
        if error_path.exists():
            raise FileExistsError(f"Refusing to overwrite error file: {error_path}")
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None
        print("No error file.")

    return output_path, error_path

In [15]:
planning_jsonl_path = DIRS["planning_batch_inputs"] / f"simplestrat5_planning_batch_input__{RUN_ID}.jsonl"

make_openai_responses_batch_jsonl(
    plan_df=planning_plan_df,
    batch_jsonl_path=planning_jsonl_path,
)

planning_batch_info = submit_openai_batch(
    batch_jsonl_path=planning_jsonl_path,
    stage_name="simplestrat5_planning",
    plan_path=planning_plan_path,
    manifest_dir=DIRS["planning_manifests"],
)

planning_batch_info

Wrote batch input: ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/batch_inputs/simplestrat5_planning_batch_input__20260523_124432__a2167636.jsonl
Requests: 12
Submitted batch:
{
  "run_id": "20260523_124432__a2167636",
  "experiment_id": "openai_followup_simplestrat5_g2css3",
  "stage": "simplestrat5_planning",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a11deaf198481909e917229c663d844",
  "input_file_id": "file-5oVd75SMQ1iE6T8pzp6ymL",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-23T17:06:55.860060+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/batch_inputs/simplestrat5_planning_batch_input__20260523_124432__a2167636.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run

{'run_id': '20260523_124432__a2167636',
 'experiment_id': 'openai_followup_simplestrat5_g2css3',
 'stage': 'simplestrat5_planning',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a11deaf198481909e917229c663d844',
 'input_file_id': 'file-5oVd75SMQ1iE6T8pzp6ymL',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-23T17:06:55.860060+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/batch_inputs/simplestrat5_planning_batch_input__20260523_124432__a2167636.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/plans/simplestrat5_planning_plan__20260523_124432__a2167636.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636',
 'manifest_path': 'ai_data/defle

In [18]:
# If the kernel restarted, uncomment and paste the manifest path:
# planning_batch_info = read_json(Path("ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_.../02_simplestrat_planning/manifests/simplestrat5_planning__batch_manifest__batch_....json"))

planning_status = check_openai_batch(planning_batch_info["batch_id"])

{
  "batch_id": "batch_6a11deaf198481909e917229c663d844",
  "status": "completed",
  "request_counts": {
    "total": 12,
    "completed": 12,
    "failed": 0
  },
  "output_file_id": "file-11U4G4ke1FReV7FFVL9LLk",
  "error_file_id": null,
  "created_at": 1779556015,
  "in_progress_at": 1779556077,
  "finalizing_at": 1779556147,
  "completed_at": 1779556149,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "usage": {
    "input_tokens": 6145,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 7602,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 13747
  }
}


In [19]:
planning_output_path, planning_error_path = download_openai_batch_results(
    batch_id=planning_batch_info["batch_id"],
    raw_output_dir=DIRS["planning_raw_outputs"],
    raw_error_dir=DIRS["planning_raw_errors"],
    stage_name="simplestrat5_planning",
)

planning_output_path, planning_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/raw_outputs/simplestrat5_planning__batch_6a11deaf198481909e917229c663d844__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/raw_outputs/simplestrat5_planning__batch_6a11deaf198481909e917229c663d844__output.jsonl'),
 None)

In [20]:
def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def flatten_usage(usage: Optional[dict]) -> dict:
    usage = usage or {}
    input_details = usage.get("input_tokens_details") or {}
    output_details = usage.get("output_tokens_details") or {}

    return {
        "usage_input_tokens": usage.get("input_tokens"),
        "usage_output_tokens": usage.get("output_tokens"),
        "usage_total_tokens": usage.get("total_tokens"),
        "usage_cached_tokens": input_details.get("cached_tokens"),
        "usage_reasoning_tokens": output_details.get("reasoning_tokens"),
    }


def parse_openai_batch_output(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    stage_name: str,
    batch_id: str,
) -> dict:
    if batch_output_path is None or not Path(batch_output_path).exists():
        raise FileNotFoundError(f"Missing batch output path: {batch_output_path}")

    plan_df = pd.read_csv(plan_path)
    plan_by_key = {row["request_key"]: row.to_dict() for _, row in plan_df.iterrows()}

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{stage_name}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{stage_name}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{stage_name}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        response = rec.get("response") or {}
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = clean_model_text(extract_text_from_responses_api_body(body))
            usage = body.get("usage") or {}

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": body.get("id"),
                "usage": usage,
                **flatten_usage(usage),
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }
        else:
            n_error += 1
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                **flatten_usage(None),
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, json_safe(record))

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "stage": stage_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{stage_name}__{batch_id}__parse_summary.json"
    write_json(summary_path, json_safe(summary))

    print(json.dumps(json_safe(summary), indent=2))
    return summary


def parse_planning_json(text: str) -> dict:
    raw = str(text).strip()
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Could not find JSON object in planning text:\n{raw[:1000]}")

    return json.loads(match.group(0))


def build_strata_table_from_planning(parsed_pkl_path: Path) -> pd.DataFrame:
    planning_df = pd.read_pickle(parsed_pkl_path)

    rows = []
    required_fields = [
        "stratum_id",
        "name",
        "description",
        "generation_instruction",
        "why_broad",
        "why_distinct",
    ]

    for _, row in planning_df.iterrows():
        if row["status"] != "success":
            raise RuntimeError(f"Planning row failed: {row.to_dict()}")

        obj = parse_planning_json(row["text"])
        strata = obj.get("strata")

        if not isinstance(strata, list):
            raise RuntimeError(f"No strata list found for task={row['task_id']}:\n{obj}")

        if len(strata) != N_STRATA:
            raise RuntimeError(f"Expected {N_STRATA} strata for task={row['task_id']}, got {len(strata)}:\n{obj}")

        seen_ids = []
        for s in strata:
            for field in required_fields:
                if not str(s.get(field, "")).strip():
                    raise RuntimeError(
                        f"Missing or empty field `{field}` for task={row['task_id']}:\n{s}"
                    )

            sid = int(s.get("stratum_id"))
            seen_ids.append(sid)

            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": row["task_id"],
                "task_family": row["task_family"],
                "task_label": row["task_label"],
                "stratum_id": sid,
                "stratum_name": str(s.get("name", "")).strip(),
                "stratum_description": str(s.get("description", "")).strip(),
                "stratum_generation_instruction": str(s.get("generation_instruction", "")).strip(),
                "why_broad": str(s.get("why_broad", "")).strip(),
                "why_distinct": str(s.get("why_distinct", "")).strip(),
                "planning_request_key": row["request_key"],
                "planning_text": row["text"],
                "planning_batch_id": row["batch_id"],
                "created_at_utc": now_iso(),
            })

        if sorted(seen_ids) != list(range(1, N_STRATA + 1)):
            raise RuntimeError(f"Bad stratum ids for task={row['task_id']}: {seen_ids}")

    out = pd.DataFrame(rows).sort_values(["task_id", "stratum_id"]).reset_index(drop=True)
    return out


planning_parse_summary = parse_openai_batch_output(
    batch_output_path=planning_output_path,
    plan_path=Path(planning_batch_info["plan_path"]),
    parsed_dir=DIRS["planning_parsed"],
    stage_name="simplestrat5_planning",
    batch_id=planning_batch_info["batch_id"],
)

strata_df = build_strata_table_from_planning(Path(planning_parse_summary["parsed_pkl_path"]))

strata_csv_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.csv"
strata_pkl_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.pkl"
strata_json_path = DIRS["planning_parsed"] / "simplestrat5_validated_strata.json"

strata_df.to_csv(strata_csv_path, index=False)
strata_df.to_pickle(strata_pkl_path)
write_json(strata_json_path, strata_df.to_dict(orient="records"))

print("Saved strata:")
print(strata_csv_path)
print(strata_pkl_path)
print(strata_json_path)

display(strata_df)

{
  "stage": "simplestrat5_planning",
  "batch_id": "batch_6a11deaf198481909e917229c663d844",
  "n_records": 12,
  "n_success": 12,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/parsed/simplestrat5_planning__batch_6a11deaf198481909e917229c663d844__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/parsed/simplestrat5_planning__batch_6a11deaf198481909e917229c663d844__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/02_simplestrat_planning/parsed/simplestrat5_planning__batch_6a11deaf198481909e917229c663d844__parsed.pkl"
}
Saved strata:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3

,provider,model,experiment_id,task_id,task_family,task_label,stratum_id,stratum_name,stratum_description,stratum_generation_instruction,why_broad,why_distinct,planning_request_key,planning_text,planning_batch_id,created_at_utc
0,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,play or exercise equipment,Use the tire or one of its parts as something ...,Generate an alternative use in the domain of p...,"This includes many kinds of games, training to...",It differs by focusing on human recreation and...,simplestrat_plan__9945be9066e3d34d5c5af9ea,"{\n ""task_id"": ""automobile_tire_alternative_u...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.858138+00:00
1,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,garden or habitat function,Use the tire or one of its parts to support pl...,Generate an alternative use related to gardeni...,There are many plausible uses involving planti...,It differs by centering on living systems and ...,simplestrat_plan__9945be9066e3d34d5c5af9ea,"{\n ""task_id"": ""automobile_tire_alternative_u...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.858149+00:00
2,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,furniture or household utility,Use the tire or one of its parts as a function...,Generate an alternative use as furniture or a ...,A tire can plausibly become many kinds of supp...,It differs by emphasizing everyday functional ...,simplestrat_plan__9945be9066e3d34d5c5af9ea,"{\n ""task_id"": ""automobile_tire_alternative_u...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.858159+00:00
3,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,4,art or visual design,Use the tire or one of its parts primarily as ...,"Generate an alternative use focused on art, de...",This supports many responses involving sculptu...,It differs by making aesthetic expression the ...,simplestrat_plan__9945be9066e3d34d5c5af9ea,"{\n ""task_id"": ""automobile_tire_alternative_u...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.858169+00:00
4,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,5,barrier or impact protection,"Use the tire or one of its parts to cushion, s...",Generate an alternative use as a protective ba...,Many valid responses can involve absorbing for...,It differs by focusing on protection and physi...,simplestrat_plan__9945be9066e3d34d5c5af9ea,"{\n ""task_id"": ""automobile_tire_alternative_u...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.858179+00:00
5,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,tool or mechanism part,Use the button as a functional component in a ...,Generate an alternative use where the button s...,"Buttons can plausibly act as spacers, knobs, r...",This focuses on physical utility in tools or m...,simplestrat_plan__09b5df154613e79056390f1a,"{\n ""task_id"": ""button_alternative_use_strata...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.857817+00:00
6,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,decorative or artistic element,"Use the button as part of visual design, ornam...",Generate an alternative use where the button f...,"Buttons vary in color, shape, texture, and arr...",This centers on aesthetic contribution rather ...,simplestrat_plan__09b5df154613e79056390f1a,"{\n ""task_id"": ""button_alternative_use_strata...",batch_6a11deaf198481909e917229c663d844,2026-05-23T17:10:54.857828+00:00
7,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,3,game or play piece,"Use the button as an object for play, rules-ba...",Generate an alternative use where the button s...,"A button can plausibly stand in for tokens, co...","This is defined by recreational use, unlike ut...",simplestrat_plan__

In [21]:
def select_css_medoid_farthest(X: np.ndarray, k: int = 3) -> list[int]:
    """
    Deterministic API-friendly approximation to G2/CSS:
    1. first anchor = medoid, i.e., most central existing response
    2. next anchors = farthest-first by max-min cosine distance
    """
    if X.shape[0] < k:
        raise ValueError(f"Need at least {k} rows, got {X.shape[0]}")

    Xn = normalize_embeddings(X)
    sim = np.clip(Xn @ Xn.T, -1.0, 1.0)
    dist = 1.0 - sim

    avg_dist = dist.mean(axis=1)
    selected = [int(np.argmin(avg_dist))]

    while len(selected) < k:
        remaining = [i for i in range(X.shape[0]) if i not in selected]
        min_dist_to_selected = dist[np.ix_(remaining, selected)].min(axis=1)
        next_idx = remaining[int(np.argmax(min_dist_to_selected))]
        selected.append(int(next_idx))

    return selected


def build_g2_css_anchor_table(
    baseline_df: pd.DataFrame,
    embeddings: np.ndarray,
    k: int = 3,
) -> pd.DataFrame:
    rows = []

    for task_id in TASK_ORDER:
        task_df = (
            baseline_df[baseline_df["task_id"].astype(str).eq(task_id)]
            .copy()
            .sort_values(["group_id", "agent_id"])
            .reset_index(drop=True)
        )

        if len(task_df) != N_FINAL_PER_TASK_METHOD_STRATEGY:
            raise RuntimeError(f"Expected 150 baseline rows for {task_id}, got {len(task_df)}")

        row_pos = task_df["_row_pos"].to_numpy(dtype=int)
        X = embeddings[row_pos]
        selected_local = select_css_medoid_farthest(X, k=k)

        for rank, local_idx in enumerate(selected_local, start=1):
            r = task_df.iloc[local_idx].to_dict()
            rows.append({
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "experiment_id": EXPERIMENT_ID,
                "task_id": task_id,
                "task_family": r.get("task_family"),
                "task_label": r.get("task_label", TASK_BY_ID[task_id]["task_label"]),
                "anchor_rank": rank,
                "selection_rule": "medoid_start_farthest_first_css",
                "baseline_row_pos": int(r["_row_pos"]),
                "baseline_group_id": r.get("group_id"),
                "baseline_agent_id": r.get("agent_id"),
                "anchor_text": clean_model_text(r["baseline_text"]),
                "created_at_utc": now_iso(),
            })

    out = pd.DataFrame(rows).sort_values(["task_id", "anchor_rank"]).reset_index(drop=True)

    counts = out.groupby("task_id").size()
    if not (counts == k).all():
        raise RuntimeError(f"Bad anchor counts:\n{counts}")

    return out


g2_anchors_df = build_g2_css_anchor_table(
    baseline_df=baseline_light,
    embeddings=experiment_data.embeddings,
    k=3,
)

g2_anchor_csv_path = DIRS["g2_processing"] / "g2_css_static3_anchors.csv"
g2_anchor_pkl_path = DIRS["g2_processing"] / "g2_css_static3_anchors.pkl"
g2_anchor_json_path = DIRS["g2_processing"] / "g2_css_static3_anchors.json"

g2_anchors_df.to_csv(g2_anchor_csv_path, index=False)
g2_anchors_df.to_pickle(g2_anchor_pkl_path)
write_json(g2_anchor_json_path, g2_anchors_df.to_dict(orient="records"))

print("Saved G2/CSS anchors:")
print(g2_anchor_csv_path)
print(g2_anchor_pkl_path)
print(g2_anchor_json_path)

display(g2_anchors_df)

Saved G2/CSS anchors:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/03_g2_css_processing/g2_css_static3_anchors.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/03_g2_css_processing/g2_css_static3_anchors.pkl
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/03_g2_css_processing/g2_css_static3_anchors.json


,provider,model,experiment_id,task_id,task_family,task_label,anchor_rank,selection_rule,baseline_row_pos,baseline_group_id,baseline_agent_id,anchor_text,created_at_utc
0,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,1,medoid_start_farthest_first_css,12692,base_047,base_047__a1,An automobile tire can be half-buried in the g...,2026-05-23T17:11:05.330571+00:00
1,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,2,medoid_start_farthest_first_css,12846,base_124,base_124__a1,Cut the tread into flexible anti-slip strips f...,2026-05-23T17:11:05.330656+00:00
2,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_automobile_tire,aut,AUT: automobile tire,3,medoid_start_farthest_first_css,12866,base_134,base_134__a1,"A tire sidewall can be cut into durable, flexi...",2026-05-23T17:11:05.330731+00:00
3,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,1,medoid_start_farthest_first_css,7436,base_119,base_119__a1,Use a button as a tiny paint palette for mixin...,2026-05-23T17:11:05.323861+00:00
4,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,2,medoid_start_farthest_first_css,7434,base_118,base_118__a1,A button can be glued under a wobbly chair leg...,2026-05-23T17:11:05.323935+00:00
5,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_button,aut,AUT: button,3,medoid_start_farthest_first_css,7404,base_103,base_103__a1,Sew a button inside a shirt collar point to ac...,2026-05-23T17:11:05.323996+00:00
6,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,1,medoid_start_farthest_first_css,9034,base_018,base_018__a1,Use the ridged edge of a key to score and snap...,2026-05-23T17:11:05.326165+00:00
7,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,2,medoid_start_farthest_first_css,9032,base_017,base_017__a1,Tape a key under a bicycle seat as an emergenc...,2026-05-23T17:11:05.326237+00:00
8,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_key,aut,AUT: key,3,medoid_start_farthest_first_css,9178,base_090,base_090__a1,Use the ridged edge of the key as a tiny saw t...,2026-05-23T17:11:05.326302+00:00
9,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,aut_shoe,aut,AUT: shoe,1,medoid_start_farthest_first_css,5668,base_135,base_135__a1,A shoe sole can be used as a textured stamp fo...,2026-05-23T17:11:05.321440+00:00


In [22]:
def build_simplestrat_r2_prompt(task: dict, strategy: str, stratum: dict) -> str:
    context = (
        "Conceptual direction assigned for this round:\n"
        f"{stratum['stratum_name']}: {stratum['stratum_description']}\n\n"
        "Additional generation constraint:\n"
        f"{stratum['stratum_generation_instruction']}\n\n"
        "Use this direction as the main conceptual path for the response. "
        "Do not mention the direction label or explain the direction.\n\n"
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="stratum")
    )


def build_g2_css_r2_prompt(task: dict, strategy: str, anchors: pd.DataFrame) -> str:
    anchors = anchors.sort_values("anchor_rank")
    if len(anchors) != 3:
        raise RuntimeError(f"Expected exactly 3 anchors for task={task['task_id']}, got {len(anchors)}")

    context = (
        "Previous responses from three other agents in the same first round:\n"
        f'1. "{anchors.iloc[0]["anchor_text"]}"\n'
        f'2. "{anchors.iloc[1]["anchor_text"]}"\n'
        f'3. "{anchors.iloc[2]["anchor_text"]}"\n\n'
    )

    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + context
        + round2_final_line_for_context(strategy, context_type="prior_responses")
    )


def slot_to_stratum_id(slot_num: int) -> int:
    # Balanced deterministic assignment: each stratum appears exactly 30 times in 150 calls.
    return ((slot_num - 1) % N_STRATA) + 1


def build_round2_new_baselines_plan(strata_df: pd.DataFrame, g2_anchors_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    strata_lookup = {
        (r.task_id, int(r.stratum_id)): r._asdict()
        for r in strata_df.itertuples(index=False)
    }

    for task in TASK_SETTINGS:
        task_id = task["task_id"]
        task_anchors = g2_anchors_df[g2_anchors_df["task_id"].astype(str).eq(task_id)].copy()

        for method in ["simplestrat5", "g2_css_static3"]:
            for strategy in STRATEGIES:
                for slot_num in range(1, N_FINAL_PER_TASK_METHOD_STRATEGY + 1):
                    slot_id = f"slot_{slot_num:03d}"

                    if method == "simplestrat5":
                        stratum_id = slot_to_stratum_id(slot_num)
                        stratum = strata_lookup[(task_id, stratum_id)]
                        user_prompt = build_simplestrat_r2_prompt(task, strategy, stratum)
                        anchor_count = 0
                        context_count = 1
                        method_label = "SimpleStrat-lite, 5 fixed auto-stratified semantic strata"
                    elif method == "g2_css_static3":
                        stratum_id = None
                        user_prompt = build_g2_css_r2_prompt(task, strategy, task_anchors)
                        anchor_count = 3
                        context_count = 3
                        method_label = "G2-inspired static CSS, 3 representative R1 examples"
                    else:
                        raise ValueError(method)

                    request_basis = {
                        "provider": PROVIDER,
                        "model": MODEL_NAME,
                        "experiment_id": EXPERIMENT_ID,
                        "round": 2,
                        "task_id": task_id,
                        "task_family": task["task_family"],
                        "task_label": task["task_label"],
                        "strategy": strategy,
                        "method": method,
                        "method_label": method_label,
                        "condition": method,
                        "slot_id": slot_id,
                        "slot_num": slot_num,
                        "stratum_id": stratum_id,
                        "anchor_count": anchor_count,
                        "context_count": context_count,
                    }

                    request_key = "r2new__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                    rows.append({
                        **request_basis,
                        "request_key": request_key,
                        "system_instructions": SYSTEM_INSTRUCTIONS,
                        "user_prompt": user_prompt,
                        "temperature": TEMPERATURE,
                        "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                        "created_at_utc": now_iso(),
                    })

    out = pd.DataFrame(rows)

    if out["request_key"].duplicated().any():
        dupes = out[out["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise RuntimeError(f"Duplicate request_key detected:\n{dupes.head()}")

    return out


round2_new_plan_df = build_round2_new_baselines_plan(
    strata_df=strata_df,
    g2_anchors_df=g2_anchors_df,
)

expected_n = len(TASK_SETTINGS) * 2 * len(STRATEGIES) * N_FINAL_PER_TASK_METHOD_STRATEGY
print("Expected R2 new-baseline requests:", expected_n)
print("Actual R2 new-baseline requests:  ", len(round2_new_plan_df))

display(
    round2_new_plan_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("request_key", "size"),
        n_strata=("stratum_id", lambda x: x.dropna().nunique()),
        n_slots=("slot_id", "nunique"),
    )
    .reset_index()
)

round2_plan_csv_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.csv"
round2_plan_pkl_path = DIRS["round2_plans"] / f"round2_new_baselines_plan__{RUN_ID}.pkl"

round2_new_plan_df.to_csv(round2_plan_csv_path, index=False)
round2_new_plan_df.to_pickle(round2_plan_pkl_path)

print("Saved:")
print(round2_plan_csv_path)
print(round2_plan_pkl_path)

Expected R2 new-baseline requests: 7200
Actual R2 new-baseline requests:   7200


,method,strategy,task_id,n,n_strata,n_slots
0,g2_css_static3,diverge,aut_automobile_tire,150,0,150
1,g2_css_static3,diverge,aut_button,150,0,150
2,g2_css_static3,diverge,aut_key,150,0,150
3,g2_css_static3,diverge,aut_shoe,150,0,150
4,g2_css_static3,diverge,aut_wooden_pencil,150,0,150
5,g2_css_static3,diverge,slogan_blood_donation,150,0,150
6,g2_css_static3,diverge,slogan_smartphone,150,0,150
7,g2_css_static3,diverge,slogan_soda,150,0,150
8,g2_css_static3,diverge,story_horror,150,0,150
9,g2_css_static3,diverge,story_jungle,150,0,150


Saved:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_124432__a2167636.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_124432__a2167636.pkl


In [23]:
for method in ["simplestrat5", "g2_css_static3"]:
    for strategy in ["vanilla", "diverge"]:
        ex = round2_new_plan_df[
            (round2_new_plan_df["method"] == method)
            & (round2_new_plan_df["strategy"] == strategy)
            & (round2_new_plan_df["task_id"] == "slogan_smartphone")
        ].iloc[0]

        print("\n" + "=" * 120)
        print(method, strategy, ex["request_key"])
        print("=" * 120)
        print(ex["user_prompt"])


simplestrat5 vanilla r2new__14ced4b37e420d4174cd83be
You are part of the marketing team at a tech company preparing to launch a new smartphone.

Generate exactly one marketing slogan for this brand-new smartphone.

Requirements:
- The slogan must not exceed 6 words.
- The slogan must be written in English.
- You may assume any detail about the smartphone.
- Do not list multiple slogans.
- Return only the slogan text.

Creativity goal:
- Make the response novel and appropriate for the task.

Conceptual direction assigned for this round:
Innovation and future: Focus the slogan on breakthrough technology, next-generation capability, or a futuristic leap forward.

Additional generation constraint:
Emphasize innovation, advanced technology, or the future.

Use this direction as the main conceptual path for the response. Do not mention the direction label or explain the direction.

Now generate one new response for the same task.

simplestrat5 diverge r2new__ac7587b05f77bb272c53eb6b
You are

In [24]:
round2_jsonl_path = DIRS["round2_batch_inputs"] / f"round2_new_baselines_batch_input__{RUN_ID}.jsonl"

make_openai_responses_batch_jsonl(
    plan_df=round2_new_plan_df,
    batch_jsonl_path=round2_jsonl_path,
)

round2_batch_info = submit_openai_batch(
    batch_jsonl_path=round2_jsonl_path,
    stage_name="round2_new_baselines",
    plan_path=round2_plan_csv_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Wrote batch input: ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/batch_inputs/round2_new_baselines_batch_input__20260523_124432__a2167636.jsonl
Requests: 7,200
Submitted batch:
{
  "run_id": "20260523_124432__a2167636",
  "experiment_id": "openai_followup_simplestrat5_g2css3",
  "stage": "round2_new_baselines",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a11dfced9a48190826db09fc37791a3",
  "input_file_id": "file-Eu12fRpUbpo3UhRtScAMQT",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-23T17:11:43.406652+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/batch_inputs/round2_new_baselines_batch_input__20260523_124432__a2167636.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run

{'run_id': '20260523_124432__a2167636',
 'experiment_id': 'openai_followup_simplestrat5_g2css3',
 'stage': 'round2_new_baselines',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a11dfced9a48190826db09fc37791a3',
 'input_file_id': 'file-Eu12fRpUbpo3UhRtScAMQT',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-23T17:11:43.406652+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/batch_inputs/round2_new_baselines_batch_input__20260523_124432__a2167636.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/plans/round2_new_baselines_plan__20260523_124432__a2167636.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636',
 'manifest_path': 'ai_data/deflect_

In [31]:
# If the kernel restarted, uncomment and paste the manifest path:
# round2_batch_info = read_json(Path("ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_.../04_round2_new_baselines/manifests/round2_new_baselines__batch_manifest__batch_....json"))

round2_status = check_openai_batch(round2_batch_info["batch_id"])

{
  "batch_id": "batch_6a11dfced9a48190826db09fc37791a3",
  "status": "completed",
  "request_counts": {
    "total": 7200,
    "completed": 7200,
    "failed": 0
  },
  "output_file_id": "file-FjAtH3sEvE7izNpgiXMcSX",
  "error_file_id": null,
  "created_at": 1779556302,
  "in_progress_at": 1779556367,
  "finalizing_at": 1779557625,
  "completed_at": 1779557908,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "usage": {
    "input_tokens": 2657400,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 778776,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 3436176
  }
}


In [32]:
round2_output_path, round2_error_path = download_openai_batch_results(
    batch_id=round2_batch_info["batch_id"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    raw_error_dir=DIRS["round2_raw_errors"],
    stage_name="round2_new_baselines",
)

round2_output_path, round2_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/raw_outputs/round2_new_baselines__batch_6a11dfced9a48190826db09fc37791a3__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/raw_outputs/round2_new_baselines__batch_6a11dfced9a48190826db09fc37791a3__output.jsonl'),
 None)

In [33]:
round2_parse_summary = parse_openai_batch_output(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    parsed_dir=DIRS["round2_parsed"],
    stage_name="round2_new_baselines",
    batch_id=round2_batch_info["batch_id"],
)

round2_new_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])

display(round2_new_df["status"].value_counts(dropna=False))

summary = (
    round2_new_df
    .groupby(["method", "strategy", "task_id"], observed=True)
    .agg(
        n=("text", "size"),
        n_success=("status", lambda x: (x == "success").sum()),
        n_nonempty=("text", lambda x: x.notna().sum()),
        input_tokens=("usage_input_tokens", "sum"),
        output_tokens=("usage_output_tokens", "sum"),
        total_tokens=("usage_total_tokens", "sum"),
    )
    .reset_index()
)

display(summary)

if not (summary["n"] == N_FINAL_PER_TASK_METHOD_STRATEGY).all():
    raise RuntimeError("Some method × strategy × task cells do not have 150 rows.")

if not (summary["n_success"] == N_FINAL_PER_TASK_METHOD_STRATEGY).all():
    bad = summary[summary["n_success"] != N_FINAL_PER_TASK_METHOD_STRATEGY]
    raise RuntimeError(f"Some cells have failed/empty outputs:\n{bad}")

compiled_csv_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}.csv"
compiled_pkl_path = DIRS["compiled"] / f"round2_new_baselines_outputs__{RUN_ID}.pkl"
summary_csv_path = DIRS["compiled"] / f"round2_new_baselines_summary__{RUN_ID}.csv"

round2_new_df.to_csv(compiled_csv_path, index=False)
round2_new_df.to_pickle(compiled_pkl_path)
summary.to_csv(summary_csv_path, index=False)

print("Saved compiled outputs:")
print(compiled_csv_path)
print(compiled_pkl_path)
print(summary_csv_path)

{
  "stage": "round2_new_baselines",
  "batch_id": "batch_6a11dfced9a48190826db09fc37791a3",
  "n_records": 7200,
  "n_success": 7200,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/parsed/round2_new_baselines__batch_6a11dfced9a48190826db09fc37791a3__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/parsed/round2_new_baselines__batch_6a11dfced9a48190826db09fc37791a3__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/04_round2_new_baselines/parsed/round2_new_baselines__batch_6a11dfced9a48190826db09fc37791a3__parsed.pkl"
}


status
success    7200
Name: count, dtype: int64

,method,strategy,task_id,n,n_success,n_nonempty,input_tokens,output_tokens,total_tokens
0,g2_css_static3,diverge,aut_automobile_tire,150,150,150,40050,3755,43805
1,g2_css_static3,diverge,aut_button,150,150,150,41400,3895,45295
2,g2_css_static3,diverge,aut_key,150,150,150,40650,3912,44562
3,g2_css_static3,diverge,aut_shoe,150,150,150,39600,3974,43574
4,g2_css_static3,diverge,aut_wooden_pencil,150,150,150,40200,4030,44230
5,g2_css_static3,diverge,slogan_blood_donation,150,150,150,33600,1652,35252
6,g2_css_static3,diverge,slogan_smartphone,150,150,150,33150,1589,34739
7,g2_css_static3,diverge,slogan_soda,150,150,150,33150,1437,34587
8,g2_css_static3,diverge,story_horror,150,150,150,141600,38809,180409
9,g2_css_static3,diverge,story_jungle,150,150,150,157650,48616,206266


Saved compiled outputs:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/05_compiled/round2_new_baselines_outputs__20260523_124432__a2167636.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/05_compiled/round2_new_baselines_outputs__20260523_124432__a2167636.pkl
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/05_compiled/round2_new_baselines_summary__20260523_124432__a2167636.csv


In [34]:
final_manifest = {
    "run_id": RUN_ID,
    "experiment_id": EXPERIMENT_ID,
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "data_root": str(DATA_ROOT),
    "baseline_pkl_path": str(baseline_pkl_path),
    "baseline_csv_path": str(baseline_csv_path),
    "planning_plan_path": str(planning_plan_path),
    "planning_batch_info": planning_batch_info,
    "planning_parse_summary": planning_parse_summary,
    "strata_csv_path": str(strata_csv_path),
    "strata_pkl_path": str(strata_pkl_path),
    "strata_json_path": str(strata_json_path),
    "g2_anchor_csv_path": str(g2_anchor_csv_path),
    "g2_anchor_pkl_path": str(g2_anchor_pkl_path),
    "g2_anchor_json_path": str(g2_anchor_json_path),
    "round2_plan_csv_path": str(round2_plan_csv_path),
    "round2_plan_pkl_path": str(round2_plan_pkl_path),
    "round2_batch_info": round2_batch_info,
    "round2_parse_summary": round2_parse_summary,
    "compiled_csv_path": str(compiled_csv_path),
    "compiled_pkl_path": str(compiled_pkl_path),
    "summary_csv_path": str(summary_csv_path),
    "created_at_utc": now_iso(),
}

final_manifest_path = DIRS["metadata"] / f"final_manifest__{RUN_ID}.json"
write_json(final_manifest_path, json_safe(final_manifest))

print("Saved final manifest:")
print(final_manifest_path)

Saved final manifest:
ai_data/deflect_creativity/openai/model_gpt-5.4/openai_followup_simplestrat5_g2css3/run_20260523_124432__a2167636/00_metadata/final_manifest__20260523_124432__a2167636.json
